In [1]:
import pandas as pd
import numpy as np
from scipy.stats import loguniform

import itertools
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV, RepeatedStratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier 

pd.set_option('display.max_colwidth', None)

## Знакомство с данными

In [2]:
df = pd.read_csv('train.tsv', header=0, sep=None, engine='python', encoding = 'utf8')
test_df = pd.read_csv('test.tsv', header=0, sep=None, engine='python', encoding = 'utf8')

In [3]:
df.head(5)

,title,is_fake
0,Москвичу Владимиру Клутину пришёл счёт за вмешательство в американские выборы,1
1,Агент Кокорина назвал езду по встречке житейской историей,0
2,Госдума рассмотрит возможность введения секретных статей Уголовного кодекса,1
3,ФАС заблокировала поставку скоростных трамваев для Москвы,0
4,Против Навального завели дело о недоносительстве на Волкова,1


In [4]:
df.dtypes

title      object
is_fake     int64
dtype: object

In [5]:
df.describe()

,is_fake
count,5758.000000
mean,0.500000
std,0.500043
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


In [6]:
df.is_fake.value_counts()

1    2879
0    2879
Name: is_fake, dtype: int64

Датасет отбаланшен абсолютно ровно, соотношение классов 50/50.  
Потребности в GridSearch и Feature Engineering не наблюдается.  
Типы данных нас вполне устроят, не будем приводить is_fake к булеву.  
Формат работы сводится к векторизации текстовых ячеек и дальнейшей классификации оных. 

## Предобработка данных

In [7]:
df.isna().sum() #проверка пропусков

title      0
is_fake    0
dtype: int64

In [8]:
df[df.duplicated(keep=False)] #посмотрим на полные дубли

,title,is_fake
402,В США зафиксировано рекордное количество банкротств,0
624,В США зафиксировано рекордное количество банкротств,0


In [9]:
df = df.drop_duplicates(ignore_index = True) #уберём дубли

In [10]:
assert (df['title'].duplicated().sum() == 0)
assert (df['title'].str.lower().duplicated().sum() == 0)

Пропусков в данных нет, а единственный дубликат безболезненно удалён.  
Он был найден проверкой на полные дубликаты, однако, последующая проверка на совпадения конкретно в столбце **title** для выявления неявных дублей пройдена. 

In [11]:
df['title_low'] = df['title'].str.lower() #создадим служебный столбик

In [12]:
df['title_low'] = df['title_low'].str.strip() #уберём пробелы

In [13]:
assert (df['title_low'].duplicated().sum() == 0) 

In [14]:
df = df.drop(labels = 'title_low', axis = 1) #уберём служебный столбец за ненадобностью

Заодно были проверены неявные дубли с пробелами в начале или конце, для этого создавался временный столбец в ловеркейсе, пробелы убирались, проверялись повторы.  
Все проверки пройдены успешно.

In [15]:
from pymystem3 import Mystem
m = Mystem()

In [16]:
df_lem = df.copy(deep=True)

In [17]:
def lemmer(row):
    return m.lemmatize(row)

In [18]:
df_lem['title'] = df_lem.title.apply(lemmer)

## Модели

In [19]:
x_train, x_test, y_train, y_test = train_test_split(df_lem['title'], df_lem['is_fake'], test_size=0.2, random_state = 451)

In [20]:
def identity_tokenizer(text):
    return text

In [21]:
tfidf1 = TfidfVectorizer(tokenizer=identity_tokenizer, max_df=1.3, lowercase=False)

Дата разбита на чанки в соотношении 80/20, в качестве векторайзера выбран хрестоматийный TFIDF.  
Попробуем зафитить данные, уже приведённые леммером в нижний регистр.  
Также подготовим почву для кросс-валидации.

In [22]:
train_vect_low, test_vect_low = tfidf1.fit_transform(x_train), tfidf1.transform(x_test)

In [23]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=451)

Объявим функции (можно было попробовать запаковать в пайплайн, но оставим так) для простого вызова и проверок алгоритмов.

In [56]:
final = []

In [25]:
def low_fit(model):
    model.fit(train_vect_low, y_train) #фит 
    
    y_pred=model.predict(test_vect_low) #предсказание
    
    score=f1_score(y_test, y_pred) 
    score_train = f1_score(y_train, model.predict(train_vect_low))       
    
    print(f'Train F1: {round(score_train*100,2)}%')
    print(f'Test F1: {round(score*100,2)}%')
    
    final.append([str(model), round(score*100,2)])

Вынесем в отдельные переменные весь текст и все ключи для обучения финальной модели.

In [26]:
full_text = tfidf1.transform(df_lem['title'])
key = df_lem['is_fake']

### Logistic Regression

Начнём с очевидной логистической регрессии, так как перед нами стоит задача классификации:

In [27]:
import warnings
warnings.filterwarnings("ignore")

In [28]:
model = LogisticRegression()
space = dict()
space['solver'] = ['newton-cg', 'lbfgs', 'liblinear','sag', 'saga']
space['C'] = loguniform(1e-5, 100)
space['penalty'] = ['l1', 'l2', 'elasticnet', 'none']

search = RandomizedSearchCV(model, space, n_iter=500, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)

print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: 0.8761731162758142
Best Hyperparameters: {'C': 15.935087812408783, 'penalty': 'l2', 'solver': 'saga'}


In [57]:
LR = LogisticRegression(C = 15.935087812408783, penalty = 'l2', solver= 'saga')

In [58]:
low_fit(LR)

Train F1: 99.96%
Test F1: 88.17%


### Random Forest

In [31]:
model = RandomForestClassifier()
space = dict()
space['criterion'] = ['gini', 'entropy']
space['n_estimators'] = np.array(range(100, 350, 25))
space['max_features'] = ['auto', 'sqrt', 'log2']


search = RandomizedSearchCV(model, space, n_iter=100, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)

print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: 0.8487673967547374
Best Hyperparameters: {'n_estimators': 175, 'max_features': 'log2', 'criterion': 'entropy'}


In [59]:
RF = RandomForestClassifier(n_estimators=175, max_features = 'log2', criterion = 'entropy', random_state = 451)

In [60]:
low_fit(RF)

Train F1: 100.0%
Test F1: 82.73%


### MLP Classifier

In [34]:
model = MLPClassifier()
space = dict()
space['alpha'] = loguniform(1e-5, 100)
space['activation'] = ['identity', 'logistic', 'tanh', 'relu']
space['solver'] = ['lbfgs', 'sgd', 'adam']
space['learning_rate'] = ['constant', 'invscaling', 'adaptive']
space['tol'] = loguniform(1e-5, 1e-2)
space['max_iter'] = np.array(range(10, 400, 5))
#итерации закомментированы для ускорения чистовой компиляции, перед которой  модели и данные не подвергались изменениям
'''
search = RandomizedSearchCV(model, space, n_iter=105, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)
# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)
'''

"\nsearch = RandomizedSearchCV(model, space, n_iter=105, scoring='f1', n_jobs=-1, cv=cv, random_state=451)\nresult = search.fit(full_text, key)\n# summarize result\nprint('Best Score: %s' % result.best_score_)\nprint('Best Hyperparameters: %s' % result.best_params_)\n"

In [35]:
MLPC = MLPClassifier(activation = 'tanh', alpha = 5.192680119733674, learning_rate = 'constant',
                    max_iter = 325, solver = 'lbfgs', tol=4.49737631126652e-05)

In [61]:
low_fit(MLPC)

Train F1: 99.98%
Test F1: 87.68%


###  Passive-Agressive Classifier

In [37]:
model = PassiveAggressiveClassifier()
space = dict()
space['C'] = loguniform(1e-5, 100)
space['max_iter'] = np.array(range(10, 10000, 10))

search = RandomizedSearchCV(model, space, n_iter=1500, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)
# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: 0.8815824361681172
Best Hyperparameters: {'C': 0.007897916617346547, 'max_iter': 4400}


In [62]:
pac=PassiveAggressiveClassifier(max_iter=4400, C=0.007897916617346547) 

In [63]:
low_fit(pac)

Train F1: 98.19%
Test F1: 88.89%


### Naive Bayes

In [40]:
model = BernoulliNB()
space = dict()
space['alpha'] = loguniform(1e-6, 100)
space['binarize'] = loguniform(1e-6, 100)
space['fit_prior'] = ['True', 'False']

search = RandomizedSearchCV(model, space, n_iter=1500, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)
# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: 0.8811109477857433
Best Hyperparameters: {'alpha': 0.29317247111137934, 'binarize': 4.1739807775421775e-06, 'fit_prior': 'False'}


In [64]:
bnb = BernoulliNB(alpha = 0.29317247111137934, fit_prior=False, binarize=4.1739807775421775e-06)

In [65]:
low_fit(bnb)

Train F1: 98.19%
Test F1: 89.34%


### KNN

In [43]:
model = KNeighborsClassifier()
space = dict()
space['n_neighbors'] = np.array(range(1,25))
space['weights'] = ['uniform', 'distance']
space['algorithm'] = ['auto', 'ball_tree', 'kd_tree', 'brute']

search = RandomizedSearchCV(model, space, n_iter=500, scoring='f1', n_jobs=-1, cv=cv, random_state=451)
result = search.fit(full_text, key)
# summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

Best Score: 0.8450800306174934
Best Hyperparameters: {'weights': 'distance', 'n_neighbors': 12, 'algorithm': 'auto'}


In [66]:
knn = KNeighborsClassifier(weights = 'distance', n_neighbors = 12, algorithm = 'auto')

In [67]:
low_fit(knn)

Train F1: 100.0%
Test F1: 87.24%


Decision Tree, Gradient Boosting показали себя слабо, выдав что-то около 75%, их не стоит даже включать в финальную таблицу. 

С учётом длины заголовков заморачиваться с эмбеддингами и NER смысла практически нет, нам не требуется семантический разбор (хотя и вполне может помочь). 

## Финальная таблица

In [46]:
test_df['lem'] = test_df.title.apply(lemmer)

In [68]:
final = pd.DataFrame(final).sort_values(by=1, ascending=False)
columns = ['model', 'f1']
final.columns = columns
final

,model,f1
4,"BernoulliNB(alpha=0.29317247111137934, binarize=4.1739807775421775e-06,\n fit_prior=False)",89.34
3,"PassiveAggressiveClassifier(C=0.007897916617346547, max_iter=4400)",88.89
0,"LogisticRegression(C=15.935087812408783, solver='saga')",88.17
2,"MLPClassifier(activation='tanh', alpha=5.192680119733674, max_iter=325,\n solver='lbfgs', tol=4.49737631126652e-05)",87.68
5,"KNeighborsClassifier(n_neighbors=12, weights='distance')",87.24
1,"RandomForestClassifier(criterion='entropy', max_features='log2',\n n_estimators=175, random_state=451)",82.73


Для финальной модели выберем самую результативную по результатам теста - Бернулли версию наивного Байеса.  
  
  
Итак, предобработанные данные, лемматизированные с помощью PyMystem и токенизированные TFIDF, будут отправленные в подготовленную на всём тестовом датасете модель, после чего будут предсказаны значения для тестового фрейма.  
Имеющиеся модели, хоть и не используют в полной мере весь потенциал NLP, как, к примеру, варианты с natasha и deeppavlov, но дают вполне хорошую точность и f1 для данной задачи. Распознать и проанализировать тексты дополнительной необходимости нет, поэтому результаты считаю приемлемыми.  
Сравнительный анализ нескольких моделей через randomsearch с кросс-валидацией позволил тюнить модели для лучшего перфоманса. Более сложные версии, особенно рекурентные сети и lstm — рассматривались в качестве варианта, но оказались слегка вне диапазона целевого моделирования: нам не требуются ни чат боты, ни генерация новостей. 
Bag of words вариации были отброшены по причине очень малой длины каждого заголовка в отдельности, а специфика (заголовки новостей) позволили пренебречь стоп-словами.  

In [69]:
final_model = BernoulliNB(alpha=0.29317247111137934, binarize=4.1739807775421775e-06, fit_prior=False)
final_model.fit(full_text, key)

BernoulliNB(alpha=0.29317247111137934, binarize=4.1739807775421775e-06,
            fit_prior=False)

In [49]:
test_token = tfidf1.transform(test_df['lem'])

In [70]:
predictions_bnb = final_model.predict(test_token)

In [51]:
predictions = test_df[['title', 'is_fake']]

In [71]:
predictions['is_fake'] = predictions_bnb

In [54]:
import csv
csv.QUOTE_NONE

3

In [72]:
predictions.to_csv('predictions.tsv', sep = ' ', index=False, doublequote=False, 
                   quoting=csv.QUOTE_NONE, quotechar="", escapechar=' ')